# Day 5 — Exercise: Heteroskedasticity & White's Repair

## Setup

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from scipy import stats
DATA_SOURCE = os.environ.get("QRC_DATA", "real")

def ols_fit(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    n = len(x)
    X = np.column_stack([np.ones(n), x])
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    s2 = e @ e / (n - X.shape[1])
    se = np.sqrt(np.diag(s2 * XtX_inv))
    return {"alpha": b[0], "beta": b[1], "se_beta": se[1], "resid": e, "X": X}

def se_white(X, e):
    XtX_inv = np.linalg.inv(X.T @ X)
    meat = X.T @ (e[:, None] ** 2 * X)
    return np.sqrt(np.diag(XtX_inv @ meat @ XtX_inv))

## E1 (L3) — The sandwich, by hand

On the heteroskedastic world below (truth β = 0.9): fit OLS, compute
the classical SE(β̂), then compute White HC0 with `se_white` above,
and White HC1 by scaling. Then confirm against statsmodels
`fit(cov_type="HC1")`. Four numbers, one table.

In [ ]:
rng = np.random.default_rng(11)
n = 1200
m = rng.normal(0, 0.011, n)
sig = 0.006 * (0.4 + 3.5 * np.abs(m) / 0.011)          # variance scales with |m|
y = 0.0001 + 0.9 * m + rng.normal(0, 1, n) * sig
# YOUR CODE

## E2 (L4) — The size tournament, repaired

Rerun day 4's world (b) — SAME generator — 600 runs. This time each
run computes BOTH classical and White-HC1 SEs. Report, for each SE
flavor: mean SE, sd of β̂ across runs (the truth it should match),
and rejection rate of the TRUE null β = 0.9 at |t| > 1.96. The table
you produce is the day's receipt: which SE held size?

In [ ]:
# YOUR CODE

## E3 (L4) — The fan, visualized and tested

On your real/synthetic market-model regression (SPY/QQQ or equivalent):
(i) scatter |e| against |x| in four |x| quartiles — print mean|e| per
quartile; (ii) run the White-test auxiliary regression e² ~ x + x² and
report t-stats. Verdict line: homoskedastic or not.

In [ ]:
# YOUR CODE

## E4 (L5) — Both flavors across ten names

For 10 stocks (real: research50 sample vs SPY; synthetic: a 10-asset
one-factor world), fit the market model and tabulate classical SE and
White SE for β, side by side, with their ratio. Cross-section
question: is the classical SE's error *systematic* (always too small)
or direction-random? Why does the answer matter for meta-research that
reads many papers' t-stats?

In [ ]:
# YOUR CODE

## E5 (L7) — The repair boundary

Your regression has BOTH heteroskedasticity (fan-shaped residuals)
AND 21-day overlapping observations. A teammate proposes: "White SEs,
we're done." Grade the proposal per assumption — which failure is
repaired, which is not, and what Monday's class (day 8) must add.

In [ ]:
# Your answer:

## Hints

- E1: HC1 = HC0 × n/(n−k) applied to the VARIANCE matrix (i.e. ×sqrt
  on the SE). statsmodels: `sm.OLS(y, sm.add_constant(m)).fit(
  cov_type="HC1")`, then `.bse`.
- E2: reuse your day-4 loop; White SEs should track the true sd of β̂
  and hold rejection near 5% ± 1.5%.
- E3: the auxiliary regression is just OLS on transformed data — your
  `ols_fit` handles e² ~ x by itself; add x² with a two-column X (or
  fit separately). A rough χ² shortcut: n·R² with 2 restrictions vs
  5% crit ≈ 5.99.
- E4: ratio White/classical mixed above and below 1 with vol-clustered
  data means time-variation dominates; consistently >1 means variance
  loads on |x| — look at your E3 quartiles to see which world yours is.